# Benchmarking Results from Classification and Regression

#### Set Up

In [1]:
import pandas as pd
import numpy as np
import site
import os

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    precision_score, recall_score, f1_score, matthews_corrcoef,
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from contextlib import nullcontext

import random

from unicodedata import bidirectional


### Utility Classes and Functions

In [3]:
def set_global_seeds(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seeds(42)

# Datasets
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class OrdinalSequenceDataset(Dataset):
    def __init__(self, X, T):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.T = torch.tensor(T, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.T[idx]

def make_cumulative_targets(y_int, K):
    y = y_int.reshape(-1, 1)
    ks = np.arange(K-1).reshape(1, -1)
    return (y > ks).astype(np.float32)

def decode_ordinal(probs, thr=0.5):
    return (probs >= thr).sum(axis=1)

# Models
class OrdinalHeadCORN(nn.Module):
    def __init__(self, in_dim, K):
        super().__init__()
        self.fc = nn.Linear(in_dim, K-1)

    def forward(self, h):
        return self.fc(h)


class OrdinalHeadCORAL(nn.Module):
    def __init__(self, in_dim, K):
        super().__init__()
        self.w = nn.Linear(in_dim, 1, bias=False)
        self._beta = nn.Parameter(torch.zeros(K-1))
        self.softplus = nn.Softplus()

    def forward(self, h):
        base = self.w(h)
        deltas = self.softplus(self._beta)
        b = torch.cumsum(deltas, dim=0)
        return base - b



class RNNHead(nn.Module):
    # Shared head:
    #   - RNN stack (LSTM/GRU, uni/bi)
    #   - BatchNorm + Dense(32, ReLU) + Dropout
    #   - Output layer (1 unit): linear (regression) or logits (classification)
    def __init__(self, input_size, rnn_type='LSTM', bidirectional=False, problem_type='classification',
                 n_classes=6, ordinal_head='coral', hidden1=128, hidden2=64, num_layers=1,
                 inter_rnn_drop=0.1, dropout=0.3, use_layernorm=False):
        super().__init__()
        self.problem_type = problem_type
        self.bidirectional = bidirectional
        self.rnn_type = rnn_type.upper()
        self.n_classes = n_classes
        self.ordinal_head = ordinal_head
        self.num_layers = int(num_layers)
        self.hidden1 = int(hidden1)
        self.hidden2 = int(hidden2)

        if self.num_layers not in (1, 2):
            raise ValueError("num_layers must be 1 or 2")

        rnn_cls = {'LSTM': nn.LSTM, 'GRU': nn.GRU}[('GRU' if 'GRU' in self.rnn_type else 'LSTM')]

        self.rnn1 = rnn_cls(
            input_size=input_size, hidden_size=self.hidden1, num_layers=1,
            batch_first=True, dropout=0.0, bidirectional=bidirectional
        )

        self.inter_rnn_drop = nn.Dropout(float(inter_rnn_drop))

        self.rnn2 = None
        if self.num_layers == 2:
            self.rnn2 = rnn_cls(
                input_size=self.hidden1*(2 if bidirectional else 1), hidden_size=self.hidden2, num_layers=1,
                batch_first=True, dropout=0.0, bidirectional=bidirectional
            )
            feat_dim = self.hidden2*(2 if bidirectional else 1)
        else:
            feat_dim = self.hidden1*(2 if bidirectional else 1)

        if use_layernorm:
            self.bn = nn.LayerNorm(feat_dim)
        else:
            self.bn = nn.BatchNorm1d(feat_dim)
        self.fc = nn.Linear(feat_dim, 32)
        self.drop = nn.Dropout(float(dropout))
        if self.problem_type == 'multiclass':
            head = self.ordinal_head.lower() if isinstance(self.ordinal_head, str) else 'coral'
            if head == 'corn':
                self.out = OrdinalHeadCORN(32, self.n_classes)
            else:
                self.out = OrdinalHeadCORAL(32, self.n_classes)
        else:
            self.out = nn.Linear(32, 1)

    def forward(self, x):
        # x: [B, T, F]
        out, _ = self.rnn1(x)
        if self.num_layers == 2:
            out = self.inter_rnn_drop(out)   # inter-layer dropout (sequence-wise)
            out, _ = self.rnn2(out)
        # take last timestep: [B, T, H] -> [B, H]
        out = out[:, -1, :]
        out = self.bn(out)
        out = F.relu(self.fc(out))
        out = self.drop(out)
        out = self.out(out)  # shape [B,1]
        return out  # regression: raw; classification: logits


def build_model(input_shape, model_type='LSTM', problem_type='regression', n_classes=6, ordinal_head='coral',
                hidden1=128, hidden2=64, num_layers=2, inter_rnn_drop=0.1, dropout=0.3, use_layernorm=False):
    seq_len, n_features = input_shape
    model_type = model_type.upper()
    kwargs = dict(
        problem_type=problem_type,
        n_classes=n_classes,
        ordinal_head=ordinal_head,
        hidden1=hidden1,
        hidden2=hidden2,
        num_layers=num_layers,
        inter_rnn_drop=inter_rnn_drop,
        dropout=dropout,
        use_layernorm=use_layernorm,
    )
    if model_type == 'LSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=False, **kwargs)
    elif model_type == 'BILSTM':
        return RNNHead(n_features, rnn_type='LSTM', bidirectional=True, **kwargs)
    elif model_type == 'GRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=False, **kwargs)
    elif model_type == 'BIGRU':
        return RNNHead(n_features, rnn_type='GRU', bidirectional=True, **kwargs)
    else:
        raise ValueError("Model type must be one of: ['LSTM','BiLSTM','GRU','BiGRU']")

# Early Stopping (PyTorch)
class EarlyStopper:
    def __init__(self, patience=15, min_delta=0.0, restore_best=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best = restore_best
        self.best_loss = float('inf')
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        improved = (self.best_loss - val_loss) > self.min_delta
        if improved:
            self.best_loss = val_loss
            self.counter = 0
            if self.restore_best:
                # Deep copy state dict
                self.best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.restore_best and self.best_state is not None:
            model.load_state_dict(self.best_state)


In [4]:
def edge_labels_from_edges(edges, decimals=1):
    labels = []
    C = len(edges) - 1
    for i in range(C):
        lo, hi = edges[i], edges[i+1]
        if i == 0:
            labels.append(f"≤ {hi*100:.{decimals}f}%")
        elif i == C - 1:
            labels.append(f"> {lo*100:.{decimals}f}%")
        else:
            labels.append(f"({lo*100:.{decimals}f}%,{hi*100:.{decimals}f}%]")
    return labels

def pct_return(series, h):
    return series.shift(-h) / series - 1.0

def safe_quantile_edges(x, n_classes=6):
    qs = np.linspace(0, 1, n_classes + 1)
    edges = np.quantile(x, qs)
    for i in range(1, len(edges)):
        if edges[i] <= edges[i-1]:
            edges[i] = np.nextafter(edges[i-1], np.inf)
    return edges

def bucketize_with_edges(x, edges):
    inner = edges[1:-1]
    return np.digitize(x, inner, right=True).astype(int)

@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval()
    chunks = []
    for xb, _ in loader:
        xb = xb.to(device)
        chunks.append(model(xb).detach().cpu())
    return torch.cat(chunks, dim=0)

def find_taus_per_threshold(Z_val, y_val_idx, grid=np.linspace(0, 1, 100)):
    if isinstance(Z_val, torch.Tensor):
        Z_val = Z_val.numpy()
    P_val = 1.0 / (1.0 + np.exp(-Z_val))
    P_rep = monotone_repair_numpy(P_val)
    K_1 = P_rep.shape[1]
    best_taus = np.full(K_1, 0.5, dtype=np.float32)
    for k in range(K_1):
        best_f1, best_tau = -1.0, 0.5
        for tau in grid:
            y_hat = decode_ordinal_with_taus(P_rep, taus_override={k: tau})
            f1 = f1_score(y_val_idx, y_hat, average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1, best_tau = f1, tau
        best_taus[k] = best_tau
    return best_taus

def monotone_repair_numpy(P):
    P = np.asarray(P).copy()
    for k in range(P.shape[1] - 2, -1, -1):
        P[:, k] = np.maximum(P[:, k], P[:, k+1])
    return P

def decode_ordinal_with_taus(P_rep, taus=None, taus_override=None):
    N, K_1 = P_rep.shape
    if taus is None:
        taus = np.full(K_1, 0.5, dtype=np.float32)
    if taus_override:
        taus = taus.copy()
        for k, v in taus_override.items():
            taus[k] = v
    comp = (P_rep >= taus.reshape(1, -1)).astype(np.int32)
    return comp.sum(axis=1).astype(np.int64)

def ordinal_to_class_probs(P_rep):
    N, K_1 = P_rep.shape
    K = K_1 + 1
    Pc = np.empty((N, K), dtype=np.float32)
    Pc[:, 0] = 1.0 - P_rep[:, 0]
    for c in range(1, K - 1):
        Pc[:, c] = np.clip(P_rep[:, c-1] - P_rep[:, c], 0.0, 1.0)
    Pc[:, K - 1] = P_rep[:, K_1 - 1]
    s = Pc.sum(axis=1, keepdims=True)
    return Pc / np.maximum(s, 1e-8)


def best_threshold_from_val(y_true, y_scores, metric='f1', grid=None):
    """
    Sweep probability thresholds on validation scores to maximize a metric.
    metric can be 'f1', 'mcc', 'accuracy', or a callable(y_true,y_pred)->float.
    Returns (best_threshold, best_metric_value).
    """
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    metric_fn = None
    if callable(metric):
        metric_fn = metric
    else:
        name = str(metric).lower()
        if name == 'f1':
            metric_fn = lambda yt, yp: f1_score(yt, yp, zero_division=0)
        elif name == 'mcc':
            metric_fn = lambda yt, yp: matthews_corrcoef(yt, yp)
        elif name in ('acc', 'accuracy'):
            metric_fn = lambda yt, yp: (yt == yp).mean()
        else:
            raise ValueError(f"Unsupported metric '{metric}'")
    best_thr = 0.5
    best_val = -np.inf
    for thr in grid:
        preds = (y_scores >= thr).astype(int)
        val = metric_fn(y_true, preds)
        if val > best_val + 1e-12 or (abs(val - best_val) <= 1e-12 and thr < best_thr):
            best_val = float(val)
            best_thr = float(thr)
    return best_thr, best_val


## Stock Prediction Pipeline

In [5]:
class StockPredictionPipeline:
    def __init__(self, df, feature_columns, model_type='LSTM', sequence_length=24, problem_type='regression', horizon_steps=1, n_classes=6, ordinal_head='coral', fixed_bucket_edges=None,
                 hidden1=256, hidden2=64, num_layers=1, inter_rnn_drop=0.0, dropout=0.4,
                 batch_size=32, learning_rate=7e-3, weight_decay=2e-3, lr_patience=7, lr_factor=0.5,
                 early_stopping_patience=20, max_epochs=20, use_layernorm=False, huber_delta=1.0, early_stopping_min_delta=0.0):
        self.df = df.copy()
        self.feature_columns = feature_columns
        self.model_type = model_type
        self.sequence_length = sequence_length
        self.problem_type = problem_type
        self.horizon_steps = horizon_steps
        self.results = []
        self.loss_curves = []
        self.n_classes = n_classes
        self.ordinal_head = ordinal_head
        self.hidden1 = hidden1
        self.hidden2 = hidden2
        self.num_layers = num_layers
        self.inter_rnn_drop = inter_rnn_drop
        self.dropout = dropout
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.lr_patience = lr_patience
        self.lr_factor = lr_factor
        self.early_stopping_patience = early_stopping_patience
        self.max_epochs = max_epochs
        self.use_layernorm = use_layernorm
        self.huber_delta = huber_delta
        self.early_stopping_min_delta = early_stopping_min_delta
        self.fixed_bucket_edges = None
        if fixed_bucket_edges is not None:
            edges = np.asarray(fixed_bucket_edges, dtype=float)
            if edges.ndim != 1:
                raise ValueError("fixed_bucket_edges must be a 1D sequence of monotonically increasing numbers")
            if edges.size < 2:
                raise ValueError("fixed_bucket_edges must contain at least two values")
            if np.any(np.diff(edges) <= 0):
                raise ValueError("fixed_bucket_edges must be strictly increasing")
            self.n_classes = int(edges.size - 1)
            self.fixed_bucket_edges = edges

        # Validate
        self._validate_inputs()

        # Device & precision
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.mixed_precision = torch.cuda.is_available()

        print(f"Pipeline initialized for a '{self.problem_type}' problem "
              f"with horizon {self.horizon_steps} steps. Device: {self.device}")

    def _validate_inputs(self):
        missing_cols = [col for col in self.feature_columns if col not in self.df.columns]
        if missing_cols:
            raise ValueError(f"Missing feature columns: {missing_cols}")

        if 'close' not in self.df.columns and 'close_price' not in self.df.columns:
            raise ValueError("No 'close' or 'close_price' column found in data")

        valid_models = ['LSTM', 'BiLSTM', 'GRU', 'BiGRU']
        if self.model_type not in valid_models:
            raise ValueError(f"Model type must be one of: {valid_models}")

        if self.problem_type not in ['regression', 'classification', 'multiclass']:
            raise ValueError("Problem type must be 'regression', 'classification', or 'multiclass'")

    def create_target_variable(self, company_data):
        company_data = company_data.copy()
        price_col = 'close' if 'close' in company_data.columns else 'close_price'
        if 'date' in company_data.columns:
            company_data = company_data.sort_values('date')
            
        h = self.horizon_steps

        company_data['target_regression'] = (
            np.log(company_data[price_col].shift(-h)) - np.log(company_data[price_col])
        )
        company_data['target_direction'] = (company_data['target_regression'] > 0).astype(int)
        company_data['ret_h'] = pct_return(company_data[price_col], h)
        if self.problem_type == 'multiclass':
            company_data = company_data.dropna(subset=['ret_h'])
        else:
            company_data = company_data.dropna()
        return company_data

    def create_sequences(self, features, *targets):
        X = []
        y_sequences = [[] for _ in targets]
        for i in range(self.sequence_length, len(features)):
            X.append(features[i-self.sequence_length:i])
            for j, target in enumerate(targets):
                y_sequences[j].append(target[i])
        return (np.array(X),) + tuple(np.array(y) for y in y_sequences)

    def _train_one_epoch(self, model, loader, optimizer, loss_fn, scaler):
        model.train()
        total_loss = 0.0
        for xb, yb in loader:
            xb = xb.to(self.device)
            yb = yb.to(self.device).view(-1, 1)

            optimizer.zero_grad(set_to_none=True)

            ctx = torch.amp.autocast('cuda') if self.mixed_precision else nullcontext()
            with ctx:
                logits = model(xb)
                loss = loss_fn(logits, yb)

            if self.mixed_precision:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * xb.size(0)

        return total_loss / len(loader.dataset)

    @torch.no_grad()
    def _eval_one_epoch(self, model, loader, loss_fn):
        model.eval()
        total_loss = 0.0
        for xb, yb in loader:
            xb = xb.to(self.device)
            yb = yb.to(self.device).view(-1, 1)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            total_loss += loss.item() * xb.size(0)
        return total_loss / len(loader.dataset)

    def _train_one_epoch_multiclass(self, model, loader, optimizer, scaler, *, pos_weight=None):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb = xb.to(self.device)
            yb = yb.to(self.device)
            optimizer.zero_grad(set_to_none=True)
            ctx = torch.amp.autocast('cuda') if self.mixed_precision else nullcontext()
            with ctx:
                logits = model(xb)
                bces = []
                for k in range(logits.shape[1]):
                    w = None if pos_weight is None else pos_weight[k]
                    bce_k = F.binary_cross_entropy_with_logits(logits[:, k], yb[:, k], pos_weight=w)
                    bces.append(bce_k)
                loss = torch.stack(bces).mean()
            if self.mixed_precision:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += float(loss.item()) * xb.size(0)
        return total / len(loader.dataset)

    @torch.no_grad()
    def _eval_one_epoch_multiclass(self, model, loader, pos_weight=None):
        model.eval()
        total = 0.0
        for xb, yb in loader:
            xb = xb.to(self.device)
            yb = yb.to(self.device)
            logits = model(xb)
            bces = []
            for k in range(logits.shape[1]):
                w = None if pos_weight is None else pos_weight[k]
                bce_k = F.binary_cross_entropy_with_logits(logits[:, k], yb[:, k], pos_weight=w)
                bces.append(bce_k)
            loss = torch.stack(bces).mean()
            total += float(loss.item()) * xb.size(0)
        return total / len(loader.dataset)

    @torch.no_grad()
    def _predict(self, model, loader):
        model.eval()
        outs = []
        for xb, _ in loader:
            xb = xb.to(self.device)
            logits = model(xb).squeeze(1).detach().cpu().numpy()
            outs.append(logits)
        return np.concatenate(outs, axis=0)

    def build_model(self, input_shape):
        model = build_model(
            input_shape,
            model_type=self.model_type,
            problem_type=self.problem_type,
            n_classes=self.n_classes,
            ordinal_head=self.ordinal_head,
            hidden1=self.hidden1,
            hidden2=self.hidden2,
            num_layers=self.num_layers,
            inter_rnn_drop=self.inter_rnn_drop,
            dropout=self.dropout,
            use_layernorm=self.use_layernorm
        )
        return model.to(self.device)

    def process_company(self, company_name, company_data, sector):
        print(f"\nProcessing {company_name} ({sector})...")
        try:
            company_data = self.create_target_variable(company_data)

            # Min samples requirement (same heuristic)
            min_samples = self.sequence_length + 75 + self.horizon_steps
            if len(company_data) < min_samples:
                print(f"Insufficient data for {company_name} ({len(company_data)} < {min_samples}). Skipping...")
                return None

            if company_data[self.feature_columns].isnull().any().any():
                print(f"Missing values in features for {company_name}. Skipping...")
                return None

            features = company_data[self.feature_columns].values
            target_reg = company_data['target_regression'].values
            target_dir = company_data['target_direction'].values

            # Create sequences
            X_raw, y_reg, y_dir = self.create_sequences(features, target_reg, target_dir)

            # TimeSeriesSplit
            n_splits = min(5, len(X_raw) // 50)
            if n_splits < 3:
                print(f"Insufficient data for proper time series validation for {company_name}. Skipping...")
                return None

            tscv = TimeSeriesSplit(n_splits=n_splits)
            splits = list(tscv.split(X_raw))
            train_idx, test_idx = splits[-1]

            # Train/Val split (last 20% of train for val)
            val_size = int(0.2 * len(train_idx))
            if val_size == 0:
                print(f'Insufficient data for validation split for {company_name}. Skipping...')
                return None
            final_train_idx = train_idx[:-val_size]
            val_idx = train_idx[-val_size:]
            
            if self.horizon_steps > 1:
                print("Adjusting for multi-step horizon...")
                gap = self.horizon_steps
                if len(final_train_idx) > gap:
                    final_train_idx = final_train_idx[:-gap]  # drop last h labels from train
                if len(val_idx) > gap:
                    val_idx = val_idx[gap:]  # drop last h labels from val
            if len(final_train_idx) == 0 or len(val_idx) == 0:
                print(f'Insufficient data after horizon adjustment for {company_name}. Skipping...')
                return None

            X_train_raw, X_val_raw, X_test_raw = X_raw[final_train_idx], X_raw[val_idx], X_raw[test_idx]
            
            F = X_raw.shape[-1]
            feat_scaler = StandardScaler()
            X_train = feat_scaler.fit_transform(X_train_raw.reshape(-1, F)).reshape(X_train_raw.shape)
            X_val   = feat_scaler.transform(X_val_raw.reshape(-1, F)).reshape(X_val_raw.shape)
            X_test  = feat_scaler.transform(X_test_raw.reshape(-1, F)).reshape(X_test_raw.shape)

            if self.problem_type == 'multiclass':
                if 'ret_h' not in company_data.columns:
                    raise RuntimeError("Expected 'ret_h' for ordinal targets but it was missing.")
                ret_full = company_data['ret_h'].values
                ret_seq_full = ret_full[self.sequence_length:]

                ret_train = ret_seq_full[final_train_idx]
                ret_val = ret_seq_full[val_idx]
                ret_test = ret_seq_full[test_idx]

                if self.fixed_bucket_edges is not None:
                    edges = self.fixed_bucket_edges
                    if int(edges.shape[0] - 1) != self.n_classes:
                        raise ValueError("fixed_bucket_edges length must match n_classes+1")
                else:
                    edges = safe_quantile_edges(ret_train, n_classes=self.n_classes)
                edges = np.asarray(edges, dtype=float)
                K = int(edges.shape[0] - 1)
                label_names = edge_labels_from_edges(edges, decimals=1)

                y_bucket = bucketize_with_edges(ret_seq_full, edges).astype(np.int64)
                y_train = y_bucket[final_train_idx]
                y_val = y_bucket[val_idx]
                y_test = y_bucket[test_idx]

                T_train = make_cumulative_targets(y_train.astype(np.int64), K)
                T_val = make_cumulative_targets(y_val.astype(np.int64), K)
                T_test = make_cumulative_targets(y_test.astype(np.int64), K)

                train_ds = OrdinalSequenceDataset(X_train, T_train)
                val_ds = OrdinalSequenceDataset(X_val, T_val)
                test_ds = OrdinalSequenceDataset(X_test, T_test)

                train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=False, drop_last=False, num_workers=0)
                val_loader = DataLoader(val_ds, batch_size=self.batch_size, shuffle=False, drop_last=False, num_workers=0)
                test_loader = DataLoader(test_ds, batch_size=self.batch_size, shuffle=False, drop_last=False, num_workers=0)

                model = self.build_model((self.sequence_length, len(self.feature_columns)))

                pos_rate = T_train.mean(axis=0)
                pos_weight = (1.0 - pos_rate) / np.clip(pos_rate, 1e-6, 1.0)
                pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(self.device)

                optimizer = Adam(model.parameters(), lr=self.learning_rate, eps=1e-7, weight_decay=self.weight_decay)
                scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=self.lr_factor, patience=self.lr_patience, min_lr=1e-7)
                early_stopper = EarlyStopper(patience=self.early_stopping_patience, min_delta=self.early_stopping_min_delta, restore_best=True)
                scaler = torch.amp.GradScaler('cuda', enabled=self.mixed_precision)

                max_epochs = self.max_epochs
                best_val = float('inf')
                epochs_trained = 0
                company_loss_rows = []

                for epoch in range(1, max_epochs + 1):
                    train_loss = self._train_one_epoch_multiclass(model, train_loader, optimizer, scaler, pos_weight=pos_weight_tensor)
                    val_loss = self._eval_one_epoch_multiclass(model, val_loader, pos_weight=pos_weight_tensor)
                    scheduler.step(val_loss)
                    stop = early_stopper.step(val_loss, model)
                    epochs_trained = epoch

                    row = {
                        'company': company_name,
                        'sector': sector,
                        'model_type': self.model_type,
                        'problem_type': self.problem_type,
                        'sequence_length': self.sequence_length,
                        'horizon_steps': self.horizon_steps,
                        'epoch': epoch,
                        'train_loss': float(train_loss),
                        'val_loss': float(val_loss),
                        'train_samples': len(X_train),
                        'val_samples': len(X_val),
                        'test_samples': len(X_test),
                    }

                    company_loss_rows.append(row)
                    self.loss_curves.append(row)

                    if epoch % 10 == 0 or stop:
                        print(f"  Epoch {epoch:03d} - train {train_loss:.5f} | val {val_loss:.5f}")

                    if stop:
                        break

                early_stopper.restore(model)

                Z_val = collect_logits(model, val_loader, self.device)
                taus = find_taus_per_threshold(Z_val, y_val.astype(np.int64))
                P_cum_val = torch.sigmoid(Z_val).cpu().numpy()
                P_rep_val = monotone_repair_numpy(P_cum_val)
                P_class_val = ordinal_to_class_probs(P_rep_val)
                mid_cut = (K // 2)
                y_val_dir = (y_val >= mid_cut).astype(int)
                prob_val_up = P_class_val[:, mid_cut:].sum(axis=1)
                dir_thr, dir_thr_score = best_threshold_from_val(y_val_dir, prob_val_up, metric='mcc')

                Z_test = collect_logits(model, test_loader, self.device)
                P_cum = torch.sigmoid(Z_test).cpu().numpy()
                P_rep = monotone_repair_numpy(P_cum)
                P_class = ordinal_to_class_probs(P_rep)

                y_pred_labels = decode_ordinal_with_taus(P_rep, taus=taus)
                y_true_labels = y_test

                labels = list(range(K))
                cm_counts = confusion_matrix(y_true_labels, y_pred_labels, labels=labels)
                cm_norm = confusion_matrix(y_true_labels, y_pred_labels, labels=labels, normalize='true')

                micro_acc = (y_true_labels == y_pred_labels).mean()
                macro_f1 = f1_score(y_true_labels, y_pred_labels, average='macro', zero_division=0)

                ret_seq_train = ret_seq_full[final_train_idx].astype(np.float32)
                mu_c = np.array([
                    ret_seq_train[y_train == c].mean() if np.any(y_train == c) else 0.0
                    for c in range(K)
                ], dtype=np.float32)

                expected_ret = (P_class * mu_c[None, :]).sum(axis=1)
                expected_ret_mean = float(expected_ret.mean())

                prob_test_up = P_class[:, mid_cut:].sum(axis=1)
                y_true_dir = (y_true_labels >= mid_cut).astype(int)
                y_pred_dir = (prob_test_up >= dir_thr).astype(int)
                precision = precision_score(y_true_dir, y_pred_dir, zero_division=0)
                recall = recall_score(y_true_dir, y_pred_dir, zero_division=0)
                f1 = f1_score(y_true_dir, y_pred_dir, zero_division=0)
                mcc = matthews_corrcoef(y_true_dir, y_pred_dir)
                directional_accuracy = (y_true_dir == y_pred_dir).mean()

                result = {
                    'company': company_name,
                    'sector': sector,
                    'model_type': self.model_type,
                    'problem_type': self.problem_type,
                    'horizon_steps': self.horizon_steps,
                    'macro_f1': macro_f1,
                    'micro_accuracy': micro_acc,
                    'expected_return_mean': expected_ret_mean,
                    'mse': np.nan,
                    'mae': np.nan,
                    'r2': np.nan,
                    'mcc': mcc,
                    'f1': f1,
                    'precision': precision,
                    'recall': recall,
                    'directional_accuracy': directional_accuracy,
                'val_directional_accuracy': val_directional_accuracy if self.problem_type == 'classification' else np.nan,
                'val_mcc': val_mcc if self.problem_type == 'classification' else np.nan,
                'val_f1': val_f1 if self.problem_type == 'classification' else np.nan,
                'val_precision': val_precision if self.problem_type == 'classification' else np.nan,
                'val_recall': val_recall if self.problem_type == 'classification' else np.nan,
                    'n_samples': int(X_raw.shape[0]),
                    'train_samples': int(X_train.shape[0]),
                    'val_samples': int(X_val.shape[0]),
                    'test_samples': int(X_test.shape[0]),
                    'epochs_trained': epochs_trained
                }
                result['confusion_matrix'] = cm_counts.tolist()
                result['confusion_matrix_normalized'] = cm_norm.tolist()
                result['bucket_edges'] = edges.tolist()
                result['bucket_labels'] = label_names
                result['taus'] = taus.astype(float).tolist()
                result['direction_threshold'] = dir_thr
                result['direction_threshold_metric'] = dir_thr_score

                print(f"  Multiclass -> Micro Acc: {micro_acc:.4f}, Macro F1: {macro_f1:.4f}, Expected Return: {expected_ret_mean:.6f}")
                print(f"  Directional threshold -> τ={dir_thr:.3f} (val F1={dir_thr_score:.4f})")

                del model
                torch.cuda.empty_cache()
                return result

            if self.problem_type == 'regression':
                y_train, y_val, y_test = y_reg[final_train_idx], y_reg[val_idx], y_reg[test_idx]
                target_scaler = StandardScaler()
                y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
                y_val_scaled   = target_scaler.transform(y_val.reshape(-1, 1)).flatten()
                train_target, val_target = y_train_scaled, y_val_scaled
            else:
                y_train, y_val, y_test = y_dir[final_train_idx], y_dir[val_idx], y_dir[test_idx]
                train_target, val_target = y_train, y_val
                target_scaler = None

            # class balance note
            if self.problem_type == 'classification':
                class_ratio = np.mean(y_train)
                if class_ratio < 0.1 or class_ratio > 0.9:
                    print(f"Severe class imbalance for {company_name} ({class_ratio:.3f}). Consider using class weights.")

            # datasets & loaders
            train_ds = SequenceDataset(X_train, train_target)
            val_ds   = SequenceDataset(X_val,   val_target)
            test_ds  = SequenceDataset(X_test,  y_test)

            train_bs = min(self.batch_size, len(train_ds))
            if train_bs < 2:
                print(f'Insufficient training samples for {company_name} (train size={len(train_ds)}). Skipping...')
                return None
            if len(train_ds) % train_bs == 1 and train_bs > 2:
                train_bs -= 1  # avoid batch size 1 for BatchNorm
            val_bs = min(self.batch_size, len(val_ds))
            test_bs = min(self.batch_size, len(test_ds))

            train_loader = DataLoader(train_ds, batch_size=train_bs, shuffle=False,  drop_last=False, num_workers=0)
            val_loader   = DataLoader(val_ds,   batch_size=val_bs,   shuffle=False, drop_last=False, num_workers=0)
            test_loader  = DataLoader(test_ds,  batch_size=test_bs,  shuffle=False, drop_last=False, num_workers=0)

            # build model
            model = self.build_model((self.sequence_length, len(self.feature_columns)))

            # loss functions
            if self.problem_type == 'regression':
                loss_fn = nn.HuberLoss(delta=self.huber_delta)
            else:
                # use BCEWithLogitsLoss for numerical stability (logits input)
                loss_fn = nn.BCEWithLogitsLoss()

            # optimizer & scheduler
            optimizer = Adam(model.parameters(), lr=self.learning_rate, eps=1e-7, weight_decay=self.weight_decay)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=self.lr_factor, patience=self.lr_patience, min_lr=1e-7)
            early_stopper = EarlyStopper(patience=self.early_stopping_patience, min_delta=self.early_stopping_min_delta, restore_best=True)
            scaler = torch.amp.GradScaler('cuda', enabled=self.mixed_precision)

            # training loop
            max_epochs = self.max_epochs
            best_val = float('inf')
            epochs_trained = 0
            company_loss_rows = []  

            for epoch in range(1, max_epochs + 1):
                train_loss = self._train_one_epoch(model, train_loader, optimizer, loss_fn, scaler)
                val_loss = self._eval_one_epoch(model, val_loader, loss_fn)
                scheduler.step(val_loss)
                stop = early_stopper.step(val_loss, model)
                epochs_trained = epoch

                
                row = {
                    'company': company_name,
                    'sector': sector,
                    'model_type': self.model_type,
                    'problem_type': self.problem_type,
                    'sequence_length': self.sequence_length,
                    'horizon_steps': self.horizon_steps,
                    'epoch': epoch,
                    'train_loss': float(train_loss),
                    'val_loss': float(val_loss),
                    'train_samples': len(X_train),
                    'val_samples': len(X_val),
                    'test_samples': len(X_test),
                }
                
                company_loss_rows.append(row)
                self.loss_curves.append(row)

                if epoch % 10 == 0 or stop:
                    print(f"  Epoch {epoch:03d} - train {train_loss:.5f} | val {val_loss:.5f}")

                if stop:
                    break

            # restore best model weights (like Keras restore_best_weights=True)
            early_stopper.restore(model)

            # summarize train/val loss for overfitting checks
            best_train_loss = np.nan
            best_val_loss = np.nan
            final_train_loss = np.nan
            final_val_loss = np.nan
            if company_loss_rows:
                best_val_loss = min(r['val_loss'] for r in company_loss_rows)
                best_train_loss = min(r['train_loss'] for r in company_loss_rows)
                final_train_loss = company_loss_rows[-1]['train_loss']
                final_val_loss = company_loss_rows[-1]['val_loss']

            # predictions
            y_pred_raw = self._predict(model, test_loader)  # raw/regression or logits

            if self.problem_type == 'regression':
                y_pred_unscaled = target_scaler.inverse_transform(y_pred_raw.reshape(-1,1)).flatten() if target_scaler is not None else y_pred_raw
                mse = mean_squared_error(y_test, y_pred_unscaled)
                mae = mean_absolute_error(y_test, y_pred_unscaled)
                r2  = r2_score(y_test, y_pred_unscaled)

                # directional metrics (derived)
                y_test_dir = (y_reg[test_idx] > 0).astype(int)
                y_pred_dir = (y_pred_unscaled > 0).astype(int)
            else:
                # logits -> probs via sigmoid -> learn best threshold on VAL
                val_logits = self._predict(model, val_loader)
                val_probs = 1.0 / (1.0 + np.exp(-val_logits))
                best_thr, best_thr_score = best_threshold_from_val(y_val, val_probs, metric='mcc')
                val_pred_dir = (val_probs >= best_thr).astype(int)
                val_precision = precision_score(y_val, val_pred_dir, zero_division=0)
                val_recall = recall_score(y_val, val_pred_dir, zero_division=0)
                val_f1 = f1_score(y_val, val_pred_dir, zero_division=0)
                val_mcc = matthews_corrcoef(y_val, val_pred_dir)
                val_directional_accuracy = (y_val == val_pred_dir).mean()
                probs = 1.0 / (1.0 + np.exp(-y_pred_raw))
                y_pred_dir = (probs >= best_thr).astype(int)
                y_test_dir = y_test
                mse = mae = r2 = np.nan

            precision = precision_score(y_test_dir, y_pred_dir, zero_division=0)
            recall    = recall_score(y_test_dir, y_pred_dir, zero_division=0)
            f1        = f1_score(y_test_dir, y_pred_dir, zero_division=0)
            mcc       = matthews_corrcoef(y_test_dir, y_pred_dir)
            directional_accuracy = np.mean(y_test_dir == y_pred_dir)

            result = {
                'company': company_name,
                'sector': sector,
                'model_type': self.model_type,
                'problem_type': self.problem_type,
                'horizon_steps': self.horizon_steps,
                'mse': mse,
                'mae': mae,
                'r2': r2,
                'mcc': mcc,
                'f1': f1,
                'precision': precision,
                'recall': recall,
                'directional_accuracy': directional_accuracy,
                'val_directional_accuracy': val_directional_accuracy if self.problem_type == 'classification' else np.nan,
                'val_mcc': val_mcc if self.problem_type == 'classification' else np.nan,
                'val_f1': val_f1 if self.problem_type == 'classification' else np.nan,
                'val_precision': val_precision if self.problem_type == 'classification' else np.nan,
                'val_recall': val_recall if self.problem_type == 'classification' else np.nan,
                'n_samples': int(X_raw.shape[0]),
                'train_samples': int(X_train.shape[0]),
                'val_samples': int(X_val.shape[0]),
                'test_samples': int(X_test.shape[0]),
                'epochs_trained': epochs_trained
            }
            if self.problem_type == 'classification':
                result['best_threshold'] = best_thr
                result['best_threshold_metric'] = best_thr_score

            if self.problem_type == 'regression':
                print(f"  Regression -> MSE: {mse:.6f}, MAE: {mae:.6f}, R²: {r2:.4f}")
            elif self.problem_type == 'classification':
                print(f"  Classification -> best τ={best_thr:.3f} (val F1={best_thr_score:.4f})")
            print(f"  Directional -> Accuracy: {directional_accuracy:.4f}, MCC: {mcc:.4f}, F1: {f1:.4f}")

            # explicit cleanup (PyTorch handles this, but keeps parity with Enrique2025)
            del model
            torch.cuda.empty_cache()

            return result

        except Exception as e:
            print(f"Error processing {company_name}: {str(e)}")
            torch.cuda.empty_cache()
            return None

    def run_pipeline(self):
        company_col = None
        for col_name in ['ticker', 'company', 'symbol']:
            if col_name in self.df.columns:
                company_col = col_name
                break
        if company_col is None:
            company_col = self.df.columns[0]
            print(f"Warning: Using '{company_col}' as company identifier column")

        companies = self.df[company_col].unique()
        print(f"Processing {len(companies)} companies with {self.model_type} model...")
        print(f"Problem type: {self.problem_type}")
        print(f"Sequence length: {self.sequence_length}")
        print(f"Features: {self.feature_columns}")

        successful_companies = 0
        for i, company in enumerate(companies, 1):
            print(f"\n[{i}/{len(companies)}] Processing {company}...")
            company_data = self.df[self.df[company_col] == company].copy()
            sector = company_data['sector'].iloc[0] if 'sector' in company_data.columns else 'Unknown'
            result = self.process_company(company, company_data, sector)
            if result:
                self.results.append(result)
                successful_companies += 1

        print(f"\n{'='*80}")
        print(f"Pipeline completed: {successful_companies}/{len(companies)} companies processed successfully")
        print(f"{'='*80}")

        if self.results:
            self.results_df = pd.DataFrame(self.results)
            return self.results_df
        else:
            print("No companies were processed successfully!")
            return pd.DataFrame()


    def analyze_results(self):
        if not hasattr(self, 'results_df') or self.results_df.empty:
            print("No results to analyze!")
            return None

        df = self.results_df
        analysis = {}

        print("" + "="*80)
        print("STOCK PREDICTION PIPELINE RESULTS")
        print("="*80)
        print(f"Model: {self.model_type} | Problem: {self.problem_type}")
        print(f"Companies analyzed: {len(df)}")
        print(f"Average samples per company: {df['n_samples'].mean():.0f}")

        print("" + "="*50)
        print("OVERALL PERFORMANCE")
        print("="*50)
        if self.problem_type == 'regression':
            print(f"Mean Squared Error:     {df['mse'].mean():.6f} (±{df['mse'].std():.6f})")
            print(f"Mean Absolute Error:    {df['mae'].mean():.6f} (±{df['mae'].std():.6f})")
            print(f"R² Score:              {df['r2'].mean():.4f} (±{df['r2'].std():.4f})")
        if self.problem_type == 'multiclass' and 'micro_accuracy' in df.columns:
            print(f"Micro Accuracy:         {df['micro_accuracy'].mean():.4f} (±{df['micro_accuracy'].std():.4f})")
            print(f"Macro F1 Score:         {df['macro_f1'].mean():.4f} (±{df['macro_f1'].std():.4f})")
            if 'expected_return_mean' in df.columns:
                print(f"Expected Return:        {df['expected_return_mean'].mean():.6f} (±{df['expected_return_mean'].std():.4f})")

        print(f"Directional Accuracy:   {df['directional_accuracy'].mean():.4f} (±{df['directional_accuracy'].std():.4f})")
        print(f"Matthews Correlation:   {df['mcc'].mean():.4f} (±{df['mcc'].std():.4f})")
        print(f"F1 Score:              {df['f1'].mean():.4f} (±{df['f1'].std():.4f})")
        print(f"Precision:             {df['precision'].mean():.4f} (±{df['precision'].std():.4f})")
        print(f"Recall:                {df['recall'].mean():.4f} (±{df['recall'].std():.4f})")

        if self.problem_type == 'multiclass' and 'expected_return_mean' in df.columns:
            print("" + "="*50)
            print("TOP 10 BY EXPECTED RETURN (mean)")
            print("="*50)
            top_er = df.nlargest(10, 'expected_return_mean')
            for _, row in top_er.iterrows():
                print(f"{row['company']:<20} | {row['sector']:<15} | E[r]_mean: {row['expected_return_mean']:.4e} | Macro-F1: {row['macro_f1']:.3f}")

        if 'sector' in df.columns and df['sector'].nunique() > 1:
            print("" + "="*50)
            print("PERFORMANCE BY SECTOR")
            print("="*50)
            sector_stats = df.groupby('sector').agg({
                'directional_accuracy': ['mean', 'std', 'count'],
                'mcc': ['mean', 'std'],
                'r2': 'mean' if self.problem_type == 'regression' else lambda x: np.nan,
                'mae': 'mean' if self.problem_type == 'regression' else lambda x: np.nan
            }).round(4)
            sector_stats.columns = ['_'.join(col).strip() if col[1] else col[0] for col in sector_stats.columns]
            sector_stats = sector_stats.sort_values('directional_accuracy_mean', ascending=False)
            for sector, row in sector_stats.iterrows():
                print(f"{sector:<20} | Acc: {row['directional_accuracy_mean']:.3f}±{row['directional_accuracy_std']:.3f} | "
                      f"MCC: {row['mcc_mean']:.3f} | Companies: {int(row['directional_accuracy_count'])}")

        print("" + "="*50)
        print("TOP 10 PERFORMERS (by Directional Accuracy)")
        print("="*50)
        top_performers = df.nlargest(10, 'directional_accuracy')
        for _, row in top_performers.iterrows():
            print(f"{row['company']:<20} | {row['sector']:<15} | "
                  f"Acc: {row['directional_accuracy']:.3f} | MCC: {row['mcc']:.3f}")

        return analysis

    def save_results(self, results, output_dir='results/benchmarking'):
        if results is not None and not results.empty:
            model_name = self.model_type

            if self.problem_type == 'regression':
                out_dir = os.path.join(output_dir, 'regression')
            elif self.problem_type == 'classification':
                out_dir = os.path.join(output_dir, 'classification')
            else:
                out_dir = os.path.join(output_dir, 'multiclass')

            os.makedirs(out_dir, exist_ok=True)

            output_path = os.path.join(out_dir, f"{model_name}.csv")

            results.to_csv(output_path, index=False)
            print(f"Results saved to {output_path}")
        else:
            print("No results to save.")
            
    def get_loss_curves_df(self):
        if not self.loss_curves:
            print("No loss curves logged yet.")
            return pd.DataFrame()
        return pd.DataFrame(self.loss_curves)

    def save_loss_curves(self, out_path='results/benchmarking/'):
        df = self.get_loss_curves_df()
        if df.empty:
            print("No loss curves to save.")
            return
        if self.problem_type == 'regression':
            out_path = os.path.join(out_path, 'regression', f"{self.model_type}_loss_curves.csv")
        elif self.problem_type == 'classification':
            out_path = os.path.join(out_path, 'classification', f"{self.model_type}_loss_curves.csv")
        else:
            out_path = os.path.join(out_path, 'multiclass', f"{self.model_type}_loss_curves.csv")
            
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        
        df.to_csv(out_path, index=False)
        print(f"Loss curves saved to {out_path}")

    def get_feature_importance_analysis(self):
        print("Feature importance analysis not implemented yet.")
        print("Consider implementing SHAP values or permutation importance for better insights.")
        return None


## Data Preparation

In [6]:
master_df = pd.read_parquet('../data/dataset/ta_nlp_sector.parquet')

In [7]:
master_df.columns

Index(['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker',
       'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
       'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
       'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv', 'ret_1d',
       'roll_ret_1d', 'roll_ret_5d', 'roll_ret_20d', 'text', 'sentiment',
       'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
       'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
       'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
       'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
       'emotion_surprize_pct', 'positive_emotion', 'negative_emotion',
       'uncertainty_emotion', 'positive_emotion_pct', 'negative_emotion_pct',
       'uncertainty_emotion_pct', 'stance_label', 'finbert_label',
       'stance_score', 'finbert_score', 'finbert_up', 'finbert_down',
       'finbert_neutral', 'sector', 'company_name', 'sec

In [8]:
master_df

,date,open,high,low,close,adj_close,volume,ticker,ema_12,ema_26,...,macd_12_26_9_sector,macdh_12_26_9_sector,macds_12_26_9_sector,rsi_14_sector,sector_bb_upper,sector_bb_middle,sector_bb_lower,market_close,sector_rel_strength,sector_dispersion_1d
0,2012-09-04,95.108574,96.448570,94.928574,96.424286,87.121140,91973000.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1464.800169,NaN,NaN
1,2012-09-05,96.510002,96.621429,95.657143,95.747147,86.509338,84093800.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,1481.232189,NaN,0.006548
2,2012-09-06,96.167145,96.898575,95.828575,96.610001,87.288956,97799100.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,28.755774,NaN,NaN,NaN,1502.627054,NaN,0.006476
3,2012-09-07,96.864288,97.497147,96.538574,97.205711,87.827171,82416600.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,28.562466,NaN,NaN,NaN,1506.971629,NaN,0.008604
4,2012-09-10,97.207146,97.612854,94.585716,94.677139,85.542564,121999500.0,AAPL,NaN,NaN,...,NaN,NaN,NaN,22.507443,NaN,NaN,NaN,1503.735325,NaN,0.012007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108587,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,77.187452,78.267858,...,-0.111192,0.016355,-0.127548,53.634155,63.049962,61.01430,58.978638,3101.328695,-0.007883,0.005228
108588,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,77.073998,78.133202,...,-0.067027,0.048417,-0.115443,54.205103,62.862103,60.94445,59.026797,3102.507723,-0.016661,0.002030
108589,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,76.924151,77.982594,...,-0.037643,0.062240,-0.099883,53.126723,62.639604,60.86875,59.097896,3128.753695,-0.017629,0.004873
108590,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,76.832744,77.860180,...,0.013408,0.090633,-0.077225,57.342960,62.525437,60.83190,59.138363,3141.201722,-0.008376,0.006185


In [9]:
columns_to_check = [
                    'sentiment',
                    
                    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
                    'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
                    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
                    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
                    'emotion_surprize_pct', 
                    
                    'positive_emotion', 'negative_emotion','uncertainty_emotion', 
                    'positive_emotion_pct', 'negative_emotion_pct','uncertainty_emotion_pct', 
                    
                    'stance_label', 'stance_score', 
                    
                    'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down',
                    'finbert_neutral', 
                    
                    'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
                    'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d',
                    'sector_ret_20d', 'sector_range', 'sector_vol_20d', 
                    
                    'ema_12_sector','ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
                    'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
                    'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
                    'market_close', 'sector_rel_strength', 'sector_dispersion_1d'
                ]

print(f"Initial master_df shape: {master_df.shape}")

master_df = master_df.dropna(subset=columns_to_check)

print(f"After dropping NaNs in selected columns, master_df shape: {master_df.shape}")

master_df.reset_index(drop=True, inplace=True)

display(master_df)

Initial master_df shape: (108592, 80)
After dropping NaNs in selected columns, master_df shape: (104476, 80)


,date,open,high,low,close,adj_close,volume,ticker,ema_12,ema_26,...,macd_12_26_9_sector,macdh_12_26_9_sector,macds_12_26_9_sector,rsi_14_sector,sector_bb_upper,sector_bb_middle,sector_bb_lower,market_close,sector_rel_strength,sector_dispersion_1d
0,2012-11-14,77.928574,78.207146,76.597145,76.697144,69.613815,119292600.0,AAPL,80.708033,84.949698,...,-0.913108,-0.181794,-0.731314,27.619972,63.948320,61.465736,58.983152,1484.350654,-0.003089,0.006800
1,2012-11-15,76.790001,77.071426,74.660004,75.088570,68.153778,197477700.0,AAPL,79.843501,84.219244,...,-0.926239,-0.155940,-0.770299,32.479352,63.646173,61.261193,58.876213,1484.574993,0.002003,0.019838
2,2012-11-16,75.028572,75.714287,72.250000,75.382858,68.420891,316723400.0,AAPL,79.157248,83.564697,...,-0.892113,-0.097452,-0.794662,37.450172,63.236926,61.078872,58.920817,1497.780485,0.010276,0.010265
3,2012-11-19,77.244286,81.071426,77.125717,80.818573,73.354591,205829400.0,AAPL,79.412836,83.361280,...,-0.733193,0.049174,-0.782368,51.350390,63.001827,61.010079,59.018330,1506.128807,0.024682,0.018850
4,2012-11-20,81.701431,81.707146,79.225716,80.129997,72.729614,160688500.0,AAPL,79.523169,83.121926,...,-0.579416,0.162361,-0.741778,53.267164,62.959435,60.996029,59.032623,1508.629302,0.025531,0.007554
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104471,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,77.187452,78.267858,...,-0.111192,0.016355,-0.127548,53.634155,63.049962,61.014300,58.978638,3101.328695,-0.007883,0.005228
104472,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,77.073998,78.133202,...,-0.067027,0.048417,-0.115443,54.205103,62.862103,60.944450,59.026797,3102.507723,-0.016661,0.002030
104473,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,76.924151,77.982594,...,-0.037643,0.062240,-0.099883,53.126723,62.639604,60.868750,59.097896,3128.753695,-0.017629,0.004873
104474,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,76.832744,77.860180,...,0.013408,0.090633,-0.077225,57.342960,62.525437,60.831900,59.138363,3141.201722,-0.008376,0.006185


In [10]:
print(master_df.columns)

Index(['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker',
       'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
       'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
       'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv', 'ret_1d',
       'roll_ret_1d', 'roll_ret_5d', 'roll_ret_20d', 'text', 'sentiment',
       'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
       'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
       'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
       'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
       'emotion_surprize_pct', 'positive_emotion', 'negative_emotion',
       'uncertainty_emotion', 'positive_emotion_pct', 'negative_emotion_pct',
       'uncertainty_emotion_pct', 'stance_label', 'finbert_label',
       'stance_score', 'finbert_score', 'finbert_up', 'finbert_down',
       'finbert_neutral', 'sector', 'company_name', 'sec

In [11]:
feature_columns = [
    'open', 'high', 'low', 'close', 'volume',
    # 'roll_ret_1d', 'roll_ret_5d', 
    # 'roll_ret_20d',
    
    'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9',
    'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3',
    'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv',
]

new_indicator_columns = [
    # 'sentiment',
                    
    # 'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
    # 'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
    # 'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
    # 'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
    # 'emotion_surprize_pct', 
    
    # 'positive_emotion', 'negative_emotion','uncertainty_emotion', 
    # 'positive_emotion_pct', 'negative_emotion_pct','uncertainty_emotion_pct', 
    
    # 'stance_label', 'stance_score', 
    
    # 'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down',
    # 'finbert_neutral', 
    
    # 'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
    # 'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d',
    # 'sector_ret_20d', 'sector_range', 'sector_vol_20d', 
    
    # 'ema_12_sector','ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
    # 'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
    # 'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
    # 'market_close', 'sector_rel_strength', 'sector_dispersion_1d'
]

# feature_columns.extend(new_indicator_columns)



sequence_length=12



all_pipelines = {}
all_results_dfs = {}
all_analyses = {}
fixed_bucket_edges = np.array([-0.08, -0.03, -0.01, 0.0, 0.01, 0.03, 0.08], dtype=float)
n_classes = len(fixed_bucket_edges) - 1
ordinal_head = 'corn'


In [12]:
print(master_df.shape)
master_df = master_df.dropna(subset=feature_columns).sort_values(['ticker','date'])
print(master_df.shape)

(104476, 80)
(104220, 80)


## Pipeline Execution

In [13]:
# print(f"\n{'='*25}\n  RUNNING PIPELINE FOR: GRU\n{'='*25}\n")

# pipeline_GRU = StockPredictionPipeline(
#     df=master_df,
#     feature_columns=feature_columns,
#     model_type='GRU',
#     sequence_length=sequence_length,
#     problem_type='classification',
#     horizon_steps=1,
#     n_classes=n_classes,
#     ordinal_head=ordinal_head,
#     fixed_bucket_edges=fixed_bucket_edges
# )

# results_GRU = pipeline_GRU.run_pipeline()

# loss_df = pipeline_GRU.get_loss_curves_df()

# pipeline_GRU.save_loss_curves('results/benchmarking/')

# if results_GRU is not None and not results_GRU.empty:
#     analysis_GRU = pipeline_GRU.analyze_results()
#     pipeline_GRU.save_results(results_GRU, output_dir='results/benchmarking/')
#     all_pipelines["GRU"] = pipeline_GRU
#     all_results_dfs["GRU"] = results_GRU
#     all_analyses["GRU"] = analysis_GRU

#     print("\nDisplaying first 5 rows of GRU results:")
#     display(results_GRU.head())
# else:
#     print(f"\n[FAILED] Pipeline for GRU did not produce any results.")

# del pipeline_GRU

In [14]:
sentinemt_columns = [
    'sentiment',
]

emotion_columns = [
    'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy',
    'emotion_neutral', 'emotion_sadness', 'emotion_surprize',
    'emotion_anger_pct', 'emotion_disgust_pct', 'emotion_fear_pct',
    'emotion_joy_pct', 'emotion_neutral_pct', 'emotion_sadness_pct',
    'emotion_surprize_pct',
]

unified_emotion_columns = [
    'positive_emotion', 'negative_emotion','uncertainty_emotion',
    'positive_emotion_pct', 'negative_emotion_pct','uncertainty_emotion_pct',
]

stance_columns = [
    'stance_label', 'stance_score',
]

finbert_columns = [
    'finbert_label', 'finbert_score', 'finbert_up', 'finbert_down',
    'finbert_neutral',
]

sector_columns = ['sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean',
                  'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d',
                  'sector_ret_20d', 'sector_range', 'sector_vol_20d', 
                  
                  'ema_12_sector','ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector',
                  'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector',
                  'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower',
                  'market_close', 'sector_rel_strength', 'sector_dispersion_1d']

In [15]:
try:
    import optuna
except ImportError:
    import sys
    !{sys.executable} -m pip install optuna
    import optuna
    
from pathlib import Path
from datetime import datetime
    
# Define feature sets to test
feature_sets = {
    'base': feature_columns,
    'sentinment' : feature_columns + sentinemt_columns,
    'emotion' : feature_columns + emotion_columns,
    'unified_emotion': feature_columns + unified_emotion_columns,
    'finbert': feature_columns + finbert_columns,
    'all_nlp': feature_columns + sentinemt_columns + emotion_columns + unified_emotion_columns + stance_columns + finbert_columns,
    'sector': feature_columns + sector_columns,
    'sector_sentiment': feature_columns + sector_columns + sentinemt_columns,
    'sector_emotion': feature_columns + sector_columns + emotion_columns,
    'sector_unified_emotion': feature_columns + sector_columns + unified_emotion_columns,
    'sector_finbert': feature_columns + sector_columns + finbert_columns,
    'sector_all_nlp': feature_columns + sector_columns + sentinemt_columns + emotion_columns + unified_emotion_columns + stance_columns + finbert_columns,
}

def objective(trial):
    params = {
        'problem_type': 'classification',  # or 'classification'
        'feature_set': trial.suggest_categorical('feature_set', list(feature_sets.keys())),
        'model_type': trial.suggest_categorical('model_type', ['LSTM', 'BiLSTM', 'GRU', 'BiGRU']),
        'sequence_length': trial.suggest_int('sequence_length', 6, 36, step=6),
        'horizon_steps': trial.suggest_categorical('horizon_steps', [1]),
        'hidden1': trial.suggest_categorical('hidden1', [64, 128, 256]),
        'hidden2': trial.suggest_categorical('hidden2', [32, 64, 128]),
        'num_layers': trial.suggest_categorical('num_layers', [1, 2]),
        'inter_rnn_drop': trial.suggest_float('inter_rnn_drop', 0.0, 0.4, step=0.1),
        'dropout': trial.suggest_float('dropout', 0.0, 0.8, step=0.1),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'learning_rate': trial.suggest_float('learning_rate', 1e-6, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
        'lr_patience': trial.suggest_categorical('lr_patience', [5, 7, 10]),
        'lr_factor': trial.suggest_categorical('lr_factor', [0.4, 0.8]),
        'early_stopping_patience': trial.suggest_categorical('early_stopping_patience', [10, 15, 20]),
        'max_epochs': trial.suggest_categorical('max_epochs', [20, 30, 50]),
        'huber_delta': trial.suggest_float('huber_delta', 0.1, 2.0),
        'early_stopping_min_delta': trial.suggest_float('early_stopping_min_delta', 0.0, 0.01),
    }

    selected_features = feature_sets[params['feature_set']]
    missing_cols = [c for c in selected_features if c not in master_df.columns]
    if missing_cols:
        print(f"Missing columns for feature_set={params['feature_set']}: {missing_cols}")
        return -1.0

    pipeline = StockPredictionPipeline(
        df=master_df,
        feature_columns=selected_features,
        model_type=params['model_type'],
        sequence_length=params['sequence_length'],
        problem_type=params['problem_type'],
        horizon_steps=params['horizon_steps'],
        n_classes=n_classes,
        ordinal_head=ordinal_head,
        fixed_bucket_edges=fixed_bucket_edges,
        hidden1=params['hidden1'],
        hidden2=params['hidden2'],
        num_layers=params['num_layers'],
        inter_rnn_drop=params['inter_rnn_drop'],
        dropout=params['dropout'],
        batch_size=params['batch_size'],
        learning_rate=params['learning_rate'],
        weight_decay=params['weight_decay'],
        lr_patience=params['lr_patience'],
        lr_factor=params['lr_factor'],
        early_stopping_patience=params['early_stopping_patience'],
        max_epochs=params['max_epochs'],
        huber_delta=params['huber_delta'],
        early_stopping_min_delta=params['early_stopping_min_delta']
    )

    results_df = pipeline.run_pipeline()
    del pipeline
    torch.cuda.empty_cache()

    if results_df is None or results_df.empty:
        print('[DEBUG] results_df empty or None')
        return -1.0

    print('[DEBUG] results_df shape:', results_df.shape)
    print('[DEBUG] results_df columns:', results_df.columns.tolist())

    # Aggregate validation metrics
    val_f1 = results_df['val_f1'].mean() if 'val_f1' in results_df.columns else np.nan
    if not np.isfinite(val_f1) and 'best_threshold_metric' in results_df.columns:
        val_f1 = results_df['best_threshold_metric'].mean()
    val_mcc = results_df['val_mcc'].mean() if 'val_mcc' in results_df.columns else np.nan
    print('[DEBUG] val_mcc:', val_mcc)
    val_precision = results_df['val_precision'].mean() if 'val_precision' in results_df.columns else np.nan
    val_recall = results_df['val_recall'].mean() if 'val_recall' in results_df.columns else np.nan
    val_dir_acc = results_df['val_directional_accuracy'].mean() if 'val_directional_accuracy' in results_df.columns else np.nan

    trial.set_user_attr('val_f1', float(val_f1))
    trial.set_user_attr('val_mcc', float(val_mcc))
    trial.set_user_attr('val_precision', float(val_precision))
    trial.set_user_attr('val_recall', float(val_recall))
    trial.set_user_attr('val_directional_accuracy', float(val_dir_acc))

    # Primary metric: mean val MCC (classification) or mean MAE (regression)
    if params['problem_type'] == 'classification':
        score = val_mcc
        if not np.isfinite(score):
            return -1.0
        return float(score)
    else:
        # minimize MAE -> maximize negative MAE
        if 'mae' not in results_df.columns:
            return -1.0
        mae = results_df['mae'].mean()
        if not np.isfinite(mae):
            return -1.0
        return float(-mae)


N_TRIALS = 500

# Run study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, timeout=60*60*12)  # 8 hours max
# Collect results
optuna_results = study.trials_dataframe()
# include user attrs
user_attrs = pd.DataFrame([t.user_attrs for t in study.trials])
optuna_results = pd.concat([optuna_results, user_attrs], axis=1)
optuna_results = optuna_results.sort_values('value', ascending=False)
optuna_results

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'../results/benchmarking/classification/optuna_tuning_sector_1H.csv'
Path('../results/benchmarking/classification').mkdir(parents=True, exist_ok=True)
optuna_results.to_csv(out_path, index=False)
print(f'Saved Optuna results to {out_path}')


[I 2026-02-22 07:31:22,456] A new study created in memory with name: no-name-9b892cf7-3f5d-4311-bc20-970fcb044548


Pipeline initialized for a 'classification' problem with horizon 1 steps. Device: cpu
Processing 88 companies with LSTM model...
Problem type: classification
Sequence length: 18
Features: ['open', 'high', 'low', 'close', 'volume', 'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9', 'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3_3', 'stochrsid_14_14_3_3', 'atrr_14', 'bb_upper', 'bb_middle', 'bb_lower', 'obv', 'sector_open_mean', 'sector_high_mean', 'sector_low_mean', 'sector_close_mean', 'sector_volume_mean', 'sector_ret_1d', 'sector_ret_5d', 'sector_ret_20d', 'sector_range', 'sector_vol_20d', 'ema_12_sector', 'ema_26_sector', 'ema_50_sector', 'macd_12_26_9_sector', 'macdh_12_26_9_sector', 'macds_12_26_9_sector', 'rsi_14_sector', 'sector_bb_upper', 'sector_bb_middle', 'sector_bb_lower', 'market_close', 'sector_rel_strength', 'sector_dispersion_1d', 'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize', 'emot

[I 2026-02-22 07:32:00,041] Trial 0 finished with value: 0.14726800993483447 and parameters: {'feature_set': 'sector_emotion', 'model_type': 'LSTM', 'sequence_length': 18, 'horizon_steps': 1, 'hidden1': 64, 'hidden2': 64, 'num_layers': 2, 'inter_rnn_drop': 0.2, 'dropout': 0.7000000000000001, 'batch_size': 32, 'learning_rate': 9.974959911029263e-06, 'weight_decay': 5.189337916096731e-06, 'lr_patience': 10, 'lr_factor': 0.8, 'early_stopping_patience': 10, 'max_epochs': 50, 'huber_delta': 0.4460295325099286, 'early_stopping_min_delta': 0.006192808604397003}. Best is trial 0 with value: 0.14726800993483447.


  Epoch 010 - train 0.71908 | val 0.71014
  Epoch 011 - train 0.73109 | val 0.70945
  Classification -> best τ=0.050 (val F1=0.0000)
  Directional -> Accuracy: 0.4754, MCC: 0.0000, F1: 0.6444

Pipeline completed: 50/88 companies processed successfully
[DEBUG] results_df shape: (50, 25)
[DEBUG] results_df columns: ['company', 'sector', 'model_type', 'problem_type', 'horizon_steps', 'mse', 'mae', 'r2', 'mcc', 'f1', 'precision', 'recall', 'directional_accuracy', 'val_directional_accuracy', 'val_mcc', 'val_f1', 'val_precision', 'val_recall', 'n_samples', 'train_samples', 'val_samples', 'test_samples', 'epochs_trained', 'best_threshold', 'best_threshold_metric']
[DEBUG] val_mcc: 0.14726800993483447
Pipeline initialized for a 'classification' problem with horizon 1 steps. Device: cpu
Processing 88 companies with LSTM model...
Problem type: classification
Sequence length: 18
Features: ['open', 'high', 'low', 'close', 'volume', 'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9', 'ma

[I 2026-02-22 07:32:35,227] Trial 1 finished with value: 0.12627030780494977 and parameters: {'feature_set': 'unified_emotion', 'model_type': 'LSTM', 'sequence_length': 18, 'horizon_steps': 1, 'hidden1': 64, 'hidden2': 32, 'num_layers': 2, 'inter_rnn_drop': 0.2, 'dropout': 0.7000000000000001, 'batch_size': 64, 'learning_rate': 0.0003806939867075793, 'weight_decay': 7.989072111016645e-07, 'lr_patience': 5, 'lr_factor': 0.4, 'early_stopping_patience': 20, 'max_epochs': 20, 'huber_delta': 1.7936697290587247, 'early_stopping_min_delta': 0.006391436586336849}. Best is trial 0 with value: 0.14726800993483447.


...

  Epoch 020 - train 0.51807 | val 1.94154
  Classification -> best τ=0.500 (val F1=0.3056)
  Directional -> Accuracy: 0.5167, MCC: 0.0000, F1: 0.0000

Pipeline completed: 48/88 companies processed successfully
[DEBUG] results_df shape: (48, 25)
[DEBUG] results_df columns: ['company', 'sector', 'model_type', 'problem_type', 'horizon_steps', 'mse', 'mae', 'r2', 'mcc', 'f1', 'precision', 'recall', 'directional_accuracy', 'val_directional_accuracy', 'val_mcc', 'val_f1', 'val_precision', 'val_recall', 'n_samples', 'train_samples', 'val_samples', 'test_samples', 'epochs_trained', 'best_threshold', 'best_threshold_metric']
[DEBUG] val_mcc: 0.28690517198933424
Pipeline initialized for a 'classification' problem with horizon 1 steps. Device: cpu
Processing 88 companies with BiGRU model...
Problem type: classification
Sequence length: 24
Features: ['open', 'high', 'low', 'close', 'volume', 'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9', 'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_

[I 2026-02-22 19:26:23,804] Trial 291 finished with value: 0.2990936924446411 and parameters: {'feature_set': 'sector', 'model_type': 'BiGRU', 'sequence_length': 24, 'horizon_steps': 1, 'hidden1': 256, 'hidden2': 32, 'num_layers': 2, 'inter_rnn_drop': 0.30000000000000004, 'dropout': 0.4, 'batch_size': 16, 'learning_rate': 0.007289840277997267, 'weight_decay': 0.0006940051514968004, 'lr_patience': 7, 'lr_factor': 0.4, 'early_stopping_patience': 15, 'max_epochs': 20, 'huber_delta': 0.8071808696450138, 'early_stopping_min_delta': 0.0007521401056699011}. Best is trial 252 with value: 0.3108162738957727.


  Epoch 016 - train 0.53692 | val 0.67810
  Classification -> best τ=0.405 (val F1=0.2732)
  Directional -> Accuracy: 0.5333, MCC: 0.0578, F1: 0.3636

Pipeline completed: 48/88 companies processed successfully
[DEBUG] results_df shape: (48, 25)
[DEBUG] results_df columns: ['company', 'sector', 'model_type', 'problem_type', 'horizon_steps', 'mse', 'mae', 'r2', 'mcc', 'f1', 'precision', 'recall', 'directional_accuracy', 'val_directional_accuracy', 'val_mcc', 'val_f1', 'val_precision', 'val_recall', 'n_samples', 'train_samples', 'val_samples', 'test_samples', 'epochs_trained', 'best_threshold', 'best_threshold_metric']
[DEBUG] val_mcc: 0.2990936924446411
Pipeline initialized for a 'classification' problem with horizon 1 steps. Device: cpu
Processing 88 companies with BiGRU model...
Problem type: classification
Sequence length: 24
Features: ['open', 'high', 'low', 'close', 'volume', 'ema_12', 'ema_26', 'ema_50', 'macd_12_26_9', 'macdh_12_26_9', 'macds_12_26_9', 'rsi_14', 'stochrsik_14_14_3

[I 2026-02-22 19:31:28,160] Trial 292 finished with value: 0.20972894853076393 and parameters: {'feature_set': 'sector', 'model_type': 'BiGRU', 'sequence_length': 24, 'horizon_steps': 1, 'hidden1': 256, 'hidden2': 32, 'num_layers': 2, 'inter_rnn_drop': 0.30000000000000004, 'dropout': 0.4, 'batch_size': 16, 'learning_rate': 7.184770372883886e-06, 'weight_decay': 0.0006690813590260184, 'lr_patience': 7, 'lr_factor': 0.4, 'early_stopping_patience': 15, 'max_epochs': 20, 'huber_delta': 0.7909262984689603, 'early_stopping_min_delta': 0.001139266704667616}. Best is trial 252 with value: 0.3108162738957727.


  Epoch 020 - train 0.68710 | val 0.68240
  Classification -> best τ=0.555 (val F1=0.2920)
  Directional -> Accuracy: 0.5167, MCC: 0.0000, F1: 0.0000

Pipeline completed: 48/88 companies processed successfully
[DEBUG] results_df shape: (48, 25)
[DEBUG] results_df columns: ['company', 'sector', 'model_type', 'problem_type', 'horizon_steps', 'mse', 'mae', 'r2', 'mcc', 'f1', 'precision', 'recall', 'directional_accuracy', 'val_directional_accuracy', 'val_mcc', 'val_f1', 'val_precision', 'val_recall', 'n_samples', 'train_samples', 'val_samples', 'test_samples', 'epochs_trained', 'best_threshold', 'best_threshold_metric']
[DEBUG] val_mcc: 0.20972894853076393
Saved Optuna results to ../results/benchmarking/classification/optuna_tuning_sector_1H.csv
